# SHROOM OOTB Test Set Baseline Runner - Colab / VS Code
Key differences from the validation runner:
- `INPUT_PATH` points to `data/SHROOM_test-labeled/test.model-agnostic.json`.
- The runner calls `run_experiment_ootb_eval.py` with `--score-split test`.
- Participant-kit checker/scorer are run **without** `--is_val`.
- Test predictions include the required `id` field.
- Each model gets an isolated `participant_submission` folder so `score.py` only sees one JSON file.

In [1]:
# 1) Configuration
from pathlib import Path
from datetime import datetime

DRIVE_PROJECT = Path("/content/drive/MyDrive/thesis_colab/model_experiments_colab")
LOCAL_PROJECT = Path("/content/model_experiments_colab")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/thesis_colab/outputs_all_baselines_test_colab_vscode_A100")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_RUN_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / RUN_TAG

SCRIPT_NAME = "run_experiment_ootb_eval.py"
INPUT_PATH = "data/SHROOM_test-labeled/test.model-agnostic.json"
REFERENCE_DIR = "data/SHROOM_test-labeled"
SCORE_SPLIT = "test"
TRACK = "model-agnostic"

WARMUP_PATH = "data/SHROOM_trial-v1.1/trial-v1.json"
WARMUP_N = 10

SEED = 42
PREVIEW_N = 2
PROMPT_VERSION = "support_prompt_v1"

CONTINUE_ON_ERROR = True
SKIP_EXISTING = True
DRY_RUN = False  # Recommended: first run with True, then set False.

# Set ONLY_TYPES = [] to run all configured OOTB model families.
ONLY_TYPES = [""]
ONLY_CONTAINS = "microsoft/deberta-xlarge-mnli"

CLEAR_LOCAL_OUTPUTS = False

print("Drive project:", DRIVE_PROJECT)
print("Local project:", LOCAL_PROJECT)
print("Drive output folder for this run:", DRIVE_RUN_OUTPUT_DIR)
print("Script:", SCRIPT_NAME)
print("Input path:", INPUT_PATH)
print("Reference dir:", REFERENCE_DIR)
print("Score split:", SCORE_SPLIT)

Drive project: /content/drive/MyDrive/thesis_colab/model_experiments_colab
Local project: /content/model_experiments_colab
Drive output folder for this run: /content/drive/MyDrive/thesis_colab/outputs_all_baselines_test_colab_vscode_A100/20260528_191326
Script: run_experiment_ootb_eval.py
Input path: data/SHROOM_test-labeled/test.model-agnostic.json
Reference dir: data/SHROOM_test-labeled
Score split: test


In [2]:
# 2) Install/check dependencies and GPU status
import subprocess
import sys
import platform

INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    packages = [
        "transformers<5",
        "accelerate",
        "sentencepiece",
        "protobuf<6",
        "scipy",
        "scikit-learn",
        "peft",
        "bitsandbytes",
        "huggingface_hub",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu_name:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("gpu_total_memory_gb:", round(props.total_memory / 1024**3, 2))
        print("bf16_supported:", torch.cuda.is_bf16_supported())
except Exception as exc:
    print("Could not inspect torch/GPU:", repr(exc))

try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not available.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.11.0+cu128
cuda_available: True
gpu_name: NVIDIA A100-SXM4-40GB
gpu_total_memory_gb: 39.49
bf16_supported: True


In [3]:
# 3) Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Could not import/mount google.colab.drive. Are you connected to a Colab runtime?")
    raise

Mounted at /content/drive


In [4]:
# 4) Sync project from Google Drive to /content
import os
import shutil
from pathlib import Path

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Drive project folder not found: {DRIVE_PROJECT}\n"
        "Create/upload your folder at this path, or edit DRIVE_PROJECT in cell 1."
    )

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

ignore = shutil.ignore_patterns(
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ipynb_checkpoints",
    "wandb",
)

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT, ignore=ignore)
os.chdir(LOCAL_PROJECT)

if CLEAR_LOCAL_OUTPUTS:
    outputs_dir = LOCAL_PROJECT / "outputs"
    if outputs_dir.exists():
        shutil.rmtree(outputs_dir)

print("Working directory:", Path.cwd())
print("Top-level files:")
for p in sorted(Path.cwd().iterdir()):
    print("-", p.name)

Working directory: /content/model_experiments_colab
Top-level files:
- data
- finetune_deberta.py
- finetune_deberta_final_eval.py
- finetune_deberta_lora_final_eval_v2.py
- finetune_deberta_lora_v2.py
- finetune_flan_lora.py
- finetune_flan_lora_final_eval.py
- finetune_flan_lora_final_eval_runtime_enhanced.py
- finetune_gemma_lora_final_eval.py
- finetune_gemma_lora_v2.py
- finetune_qwen_lora.py
- finetune_qwen_lora_final_eval.py
- participant_kit
- run_experiment.py
- run_experiment_ootb_eval.py
- run_experiment_ootb_eval_runtime_enhanced.py
- src


In [5]:
# 5) Sanity-check expected files before running
from pathlib import Path

required_paths = [
    SCRIPT_NAME,
    "src/data.py",
    "src/prompts.py",
    "src/models_flan.py",
    "src/models_deberta.py",
    "src/models_qwen.py",
    "src/models_gemma.py",
    INPUT_PATH,
    REFERENCE_DIR,
    WARMUP_PATH,
    "participant_kit/check_output.py",
    "participant_kit/score.py",
]

missing = [p for p in required_paths if not Path(p).exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print("-", p)
    raise FileNotFoundError("Project structure check failed.")

print("Project structure looks good.")

Project structure looks good.


In [6]:
# 6) Model list: OOTB baselines from the screenshot/model plan
MODELS = [
    {"model_type": "flan", "model_name": "google/flan-t5-small", "run": True, "use_4bit": False},
    {"model_type": "flan", "model_name": "google/flan-t5-base", "run": True, "use_4bit": False},
    {"model_type": "flan", "model_name": "google/flan-t5-large", "run": True, "use_4bit": False},
    {"model_type": "flan", "model_name": "google/flan-t5-xl", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "microsoft/deberta-xlarge-mnli", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "cross-encoder/nli-deberta-v3-xsmall", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "cross-encoder/nli-deberta-v3-small", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "cross-encoder/nli-deberta-v3-base", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "cross-encoder/nli-deberta-v3-large", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "sileod/deberta-v3-base-tasksource-nli", "run": True, "use_4bit": False},
    {"model_type": "deberta", "model_name": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli", "run": True, "use_4bit": False},
    {"model_type": "qwen", "model_name": "Qwen/Qwen2.5-0.5B-Instruct", "run": True, "use_4bit": False},
    {"model_type": "qwen", "model_name": "Qwen/Qwen2.5-1.5B-Instruct", "run": True, "use_4bit": False},
    {"model_type": "qwen", "model_name": "Qwen/Qwen2.5-3B-Instruct", "run": True, "use_4bit": False},
    {"model_type": "qwen", "model_name": "Qwen/Qwen2.5-7B-Instruct", "run": True, "use_4bit": False}, # Will need to run on T4 with expanded RAM
    {"model_type": "gemma", "model_name": "google/gemma-3-270m-it", "run": True, "use_4bit": False},
    {"model_type": "gemma", "model_name": "google/gemma-3-1b-it", "run": True, "use_4bit": False},
    {"model_type": "gemma", "model_name": "google/gemma-3-4b-it", "run": True, "use_4bit": False},
]

print(f"Configured {len(MODELS)} models.")
for item in MODELS:
    print(f"- {item['model_type']}: {item['model_name']} | use_4bit={item.get('use_4bit', False)}")

Configured 18 models.
- flan: google/flan-t5-small | use_4bit=False
- flan: google/flan-t5-base | use_4bit=False
- flan: google/flan-t5-large | use_4bit=False
- flan: google/flan-t5-xl | use_4bit=False
- deberta: microsoft/deberta-xlarge-mnli | use_4bit=False
- deberta: cross-encoder/nli-deberta-v3-xsmall | use_4bit=False
- deberta: cross-encoder/nli-deberta-v3-small | use_4bit=False
- deberta: cross-encoder/nli-deberta-v3-base | use_4bit=False
- deberta: cross-encoder/nli-deberta-v3-large | use_4bit=False
- deberta: sileod/deberta-v3-base-tasksource-nli | use_4bit=False
- deberta: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli | use_4bit=False
- qwen: Qwen/Qwen2.5-0.5B-Instruct | use_4bit=False
- qwen: Qwen/Qwen2.5-1.5B-Instruct | use_4bit=False
- qwen: Qwen/Qwen2.5-3B-Instruct | use_4bit=False
- qwen: Qwen/Qwen2.5-7B-Instruct | use_4bit=False
- gemma: google/gemma-3-270m-it | use_4bit=False
- gemma: google/gemma-3-1b-it | use_4bit=False
- gemma: google/gemma-3-4b-it | use_4

In [7]:
# 7) Optional Hugging Face login
LOGIN_TO_HF = False # Set True for Gemma

if LOGIN_TO_HF:
    from huggingface_hub import login

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token is None:
        import getpass
        token = getpass.getpass("Paste Hugging Face token: ")

    login(token=token)
    print("Logged into Hugging Face.")
else:
    print("HF login skipped. Set LOGIN_TO_HF = True if a model requires access.")

HF login skipped. Set LOGIN_TO_HF = True if a model requires access.


In [8]:
# 9) Runner helpers: filtering, streamed subprocess logs, and Drive backup
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime


def safe_filename(text: str) -> str:
    return (
        text.replace("/", "__")
        .replace("\\", "__")
        .replace(":", "_")
        .replace(" ", "_")
    )


def selected_models(models: list[dict]) -> list[dict]:
    only_types = {x.strip() for x in ONLY_TYPES if x.strip()} if ONLY_TYPES else set()
    only_contains = ONLY_CONTAINS.strip().lower()

    selected = []
    for item in models:
        if item.get("run", True) is False:
            continue
        if only_types and item["model_type"] not in only_types:
            continue
        if only_contains and only_contains not in item["model_name"].lower():
            continue
        selected.append(item)
    return selected


def backup_outputs_to_drive() -> None:
    DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    src_outputs = LOCAL_PROJECT / "outputs"
    dst_outputs = DRIVE_RUN_OUTPUT_DIR / "outputs"

    if src_outputs.exists():
        if dst_outputs.exists():
            shutil.rmtree(dst_outputs)
        shutil.copytree(src_outputs, dst_outputs)
        print(f"Backed up outputs -> {dst_outputs}")
    else:
        print("No outputs directory yet; skipping outputs backup.")


def run_streamed(cmd: list[str], env: dict[str, str], log_path: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Command:", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return process.wait()


def run_one_model(item: dict) -> dict:
    model_type = item["model_type"]
    model_name = item["model_name"]
    use_4bit = bool(item.get("use_4bit", False))

    metadata_path = LOCAL_PROJECT / "outputs" / "metadata" / f"run__{SCORE_SPLIT}__{safe_filename(model_name)}.json"
    if SKIP_EXISTING and metadata_path.exists():
        print(f"Skipping existing result: {model_name}")
        return {"model_name": model_name, "model_type": model_type, "status": "skipped_existing", "returncode": 0}

    notes = item.get("notes") or (
        f"Colab VS Code OOTB test baseline; run_tag={RUN_TAG}; use_4bit={use_4bit}"
    )

    cmd = [
        sys.executable,
        SCRIPT_NAME,
        "--model-type", model_type,
        "--model-name", model_name,
        "--input-path", INPUT_PATH,
        "--score-split", SCORE_SPLIT,
        "--track", TRACK,
        "--reference-dir", REFERENCE_DIR,
        "--seed", str(SEED),
        "--preview-n", str(PREVIEW_N),
        "--prompt-version", PROMPT_VERSION,
        "--notes", notes,
        "--warmup-path", WARMUP_PATH,
        "--warmup-n", str(WARMUP_N),
    ]

    env = os.environ.copy()
    env["SHROOM_USE_4BIT"] = "1" if use_4bit else "0"
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    log_path = DRIVE_RUN_OUTPUT_DIR / "logs" / f"{safe_filename(model_name)}.log"

    print("\n" + "=" * 100)
    print(f"Running TEST OOTB: {model_type} | {model_name} | use_4bit={use_4bit}")
    print("=" * 100)

    if DRY_RUN:
        print("DRY_RUN=True, not executing.")
        print("Command:", " ".join(cmd))
        return {"model_name": model_name, "model_type": model_type, "use_4bit": use_4bit, "status": "dry_run", "returncode": 0}

    start = datetime.now().isoformat(timespec="seconds")
    returncode = run_streamed(cmd, env=env, log_path=log_path)
    end = datetime.now().isoformat(timespec="seconds")

    status = "success" if returncode == 0 else "failed"

    result = {
        "model_name": model_name,
        "model_type": model_type,
        "use_4bit": use_4bit,
        "status": status,
        "returncode": returncode,
        "started_at": start,
        "ended_at": end,
        "log_path": str(log_path),
    }

    backup_outputs_to_drive()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    return result

In [9]:
# 10) Optional smoke/dry run
print("DRY_RUN:", DRY_RUN)
print("ONLY_TYPES:", ONLY_TYPES)
print("ONLY_CONTAINS:", ONLY_CONTAINS)

selected = selected_models(MODELS)
print(f"Selected {len(selected)} model(s):")
for item in selected:
    print(f"- {item['model_type']}: {item['model_name']} | use_4bit={item.get('use_4bit', False)}")

DRY_RUN: False
ONLY_TYPES: ['']
ONLY_CONTAINS: microsoft/deberta-xlarge-mnli
Selected 1 model(s):
- deberta: microsoft/deberta-xlarge-mnli | use_4bit=False


In [10]:
# 11) Run selected models and back up after each one
DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DRIVE_RUN_OUTPUT_DIR / "run_manifest.json"
results = []

selected = selected_models(MODELS)
if not selected:
    raise ValueError("No models selected. Check ONLY_TYPES / ONLY_CONTAINS / run flags.")

for item in selected:
    result = run_one_model(item)
    results.append(result)

    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "run_tag": RUN_TAG,
                "drive_project": str(DRIVE_PROJECT),
                "local_project": str(LOCAL_PROJECT),
                "drive_run_output_dir": str(DRIVE_RUN_OUTPUT_DIR),
                "script_name": SCRIPT_NAME,
                "input_path": INPUT_PATH,
                "reference_dir": REFERENCE_DIR,
                "score_split": SCORE_SPLIT,
                "track": TRACK,
                "warmup_path": WARMUP_PATH,
                "warmup_n": WARMUP_N,
                "seed": SEED,
                "results": results,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    if result["returncode"] != 0:
        print(f"FAILED: {item['model_name']} with return code {result['returncode']}")
        if not CONTINUE_ON_ERROR:
            break

print("\nBatch complete. Manifest:", manifest_path)
print(json.dumps(results, indent=2))


Running TEST OOTB: deberta | microsoft/deberta-xlarge-mnli | use_4bit=False
Command: /usr/bin/python3 run_experiment_ootb_eval.py --model-type deberta --model-name microsoft/deberta-xlarge-mnli --input-path data/SHROOM_test-labeled/test.model-agnostic.json --score-split test --track model-agnostic --reference-dir data/SHROOM_test-labeled --seed 42 --preview-n 2 --prompt-version support_prompt_v1 --notes Colab VS Code OOTB test baseline; run_tag=20260528_191326; use_4bit=False --warmup-path data/SHROOM_trial-v1.1/trial-v1.json --warmup-n 10
2026-05-28 19:15:17.924305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-28 19:15:17.994329: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in perfo

In [11]:
# 12) Summarize metadata files found in outputs/metadata
import json
from pathlib import Path

metadata_dir = LOCAL_PROJECT / "outputs" / "metadata"
rows = []

if metadata_dir.exists():
    for path in sorted(metadata_dir.glob(f"run__{SCORE_SPLIT}__*.json")):
        try:
            meta = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print("Could not read", path, exc)
            continue

        scores = meta.get("scores", {})
        direct = meta.get("direct_metrics", {})
        comp = meta.get("computational_cost", {})
        rows.append({
            "model_type": meta.get("model_type"),
            "model_name": meta.get("model_name"),
            "parameter_count": comp.get("parameter_count"),
            "score_split": meta.get("score_split"),
            "test_acc": scores.get("agnostic_acc", scores.get("acc_agnostic", direct.get("accuracy"))),
            "test_rho": scores.get("agnostic_rho", scores.get("rho_agnostic", direct.get("rho"))),
            "direct_acc": direct.get("accuracy"),
            "direct_rho": direct.get("rho"),
            "mean_latency_s": comp.get("mean_inference_latency_seconds_per_example"),
            "metadata_file": str(path),
        })

if not rows:
    print("No metadata rows found yet.")
else:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df.sort_values(["model_type", "parameter_count"], na_position="last"))
    except Exception:
        for row in rows:
            print(row)

backup_outputs_to_drive()

,model_type,model_name,parameter_count,score_split,test_acc,test_rho,direct_acc,direct_rho,mean_latency_s,metadata_file
0,deberta,microsoft/deberta-xlarge-mnli,758856707,test,0.769333,0.681578,0.769333,0.681578,0.097251,/content/model_experiments_colab/outputs/metad...


Backed up outputs -> /content/drive/MyDrive/thesis_colab/outputs_all_baselines_test_colab_vscode_A100/20260528_191326/outputs


In [12]:
# 13) Inspect score.py text outputs
scores_dir = LOCAL_PROJECT / "outputs" / "scores"
if scores_dir.exists():
    for path in sorted(scores_dir.glob(f"{SCORE_SPLIT}_scores__*.txt")):
        print("\n" + "=" * 100)
        print(path)
        print("=" * 100)
        print(path.read_text(encoding="utf-8"))
else:
    print("No scores directory yet.")


/content/model_experiments_colab/outputs/scores/test_scores__microsoft__deberta-xlarge-mnli.txt
agnostic_acc:0.7693333333333333
agnostic_rho:0.6815780133811903



In [ ]:
# 14) Optional: zip current /content outputs for manual retrieval
import shutil
from pathlib import Path

zip_base = Path("/content/shroom_all_baseline_test_outputs_colab_vscode")
zip_file = zip_base.with_suffix(".zip")

if zip_file.exists():
    zip_file.unlink()

outputs_dir = LOCAL_PROJECT / "outputs"
if outputs_dir.exists():
    shutil.make_archive(str(zip_base), "zip", root_dir=LOCAL_PROJECT, base_dir="outputs")
    print("Created:", zip_file)
else:
    print("No outputs directory to zip.")